# Notebook 10 - Team Season Stats

**Goal**: Pre-compute all team statistics used by the Teams dashboard page.

**Output**: `team_season_stats.csv` - one row per team per season covering:
- Phase scoring profile: average runs scored in powerplay, middle, and death per match
- Win/loss record and win rate
- Toss decisions and outcomes
- Batting depth: average runs from lower-order batters (positions 7-9) per match

## 1. Imports

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.3f}'.format)

## 2. Load Data

In [2]:
del_df = pd.read_csv('../data/processed/deliveries.csv')
del_df = del_df[del_df['super_over'] == False].copy()

mat_df = pd.read_csv('../data/processed/matches.csv')

print(f'Deliveries (no super overs): {len(del_df):,}')
print(f'Matches: {len(mat_df):,}')
print(f'Seasons: {del_df["season"].min()} - {del_df["season"].max()}')

Deliveries (no super overs): 295,557
Matches: 1,243
Seasons: 2008 - 2026


## 3. Normalize Team Names

Some franchises have changed names (Delhi Daredevils to Delhi Capitals, Royal Challengers Bangalore to Royal Challengers Bengaluru). I map all historical names to a single canonical name per franchise so seasons can be compared consistently.

In [3]:
# Map historical names to current canonical franchise name
TEAM_RENAME = {
    'Delhi Daredevils':              'Delhi Capitals',
    'Royal Challengers Bangalore':   'Royal Challengers Bengaluru',
    'Kings XI Punjab':               'Punjab Kings',
    'Rising Pune Supergiants':       'Rising Pune Supergiants',
    'Rising Pune Supergiant':        'Rising Pune Supergiants',
    'Pune Warriors':                 'Pune Warriors',
    'Deccan Chargers':               'Deccan Chargers',
    'Kochi Tuskers Kerala':          'Kochi Tuskers Kerala',
}

# Apply to deliveries
for col in ['batting_team', 'bowling_team']:
    del_df[col] = del_df[col].replace(TEAM_RENAME)

# Apply to matches
for col in ['team1', 'team2', 'toss_winner', 'winner']:
    mat_df[col] = mat_df[col].replace(TEAM_RENAME)

print('Active teams in 2021-2026:')
print(sorted(del_df[del_df['season'] >= 2021]['batting_team'].unique()))

Active teams in 2021-2026:
['Chennai Super Kings', 'Delhi Capitals', 'Gujarat Titans', 'Kolkata Knight Riders', 'Lucknow Super Giants', 'Mumbai Indians', 'Punjab Kings', 'Rajasthan Royals', 'Royal Challengers Bengaluru', 'Sunrisers Hyderabad']


## 4. Phase Scoring Profile

For each team and season, compute average runs scored per match in each phase. I sum runs per match-phase first, then average across matches - this gives runs per match (not per ball) which is the most intuitive metric for comparing team scoring style.

In [4]:
# Sum runs per match per phase per batting team
phase_runs = (
    del_df.groupby(['season', 'match_id', 'batting_team', 'phase'])['total_runs']
    .sum()
    .reset_index(name='match_phase_runs')
)

# Average across all matches that team played at home/away in that season
phase_avg = (
    phase_runs.groupby(['season', 'batting_team', 'phase'])['match_phase_runs']
    .mean()
    .reset_index(name='avg_runs')
)

# Pivot to wide: one column per phase
phase_wide = phase_avg.pivot_table(
    index=['season', 'batting_team'],
    columns='phase',
    values='avg_runs',
    fill_value=0.0
).reset_index()
phase_wide.columns.name = None
phase_wide = phase_wide.rename(columns={
    'powerplay': 'pp_avg_runs',
    'middle':    'mid_avg_runs',
    'death':     'death_avg_runs',
    'batting_team': 'team',
})

print('Phase scoring preview:')
print(phase_wide[phase_wide['season'] == 2024].sort_values('death_avg_runs', ascending=False).head(5))

Phase scoring preview:
     season                         team  death_avg_runs  mid_avg_runs  \
144    2024  Royal Challengers Bengaluru          58.286        82.533   
136    2024          Chennai Super Kings          53.000        74.929   
137    2024               Delhi Capitals          49.308        73.929   
145    2024          Sunrisers Hyderabad          48.933        77.812   
138    2024               Gujarat Titans          48.417        75.250   

     pp_avg_runs  
144       58.400  
136       52.357  
137       64.071  
145       67.062  
138       46.333  


## 5. Win/Loss Record Per Season

In [5]:
# Each match has team1 and team2 - create two rows (one per team) to compute records
team1 = mat_df[['season', 'match_id', 'team1', 'winner']].rename(columns={'team1': 'team'})
team2 = mat_df[['season', 'match_id', 'team2', 'winner']].rename(columns={'team2': 'team'})
all_teams = pd.concat([team1, team2], ignore_index=True)

# Exclude no-result matches (winner is NaN or the match was abandoned)
all_teams = all_teams[all_teams['winner'].notna()]

all_teams['won'] = (all_teams['team'] == all_teams['winner']).astype(int)

records = (
    all_teams.groupby(['season', 'team'])
    .agg(matches=('match_id', 'nunique'), wins=('won', 'sum'))
    .reset_index()
)
records['losses']   = records['matches'] - records['wins']
records['win_rate'] = records['wins'] / records['matches']

print('Win rate sample (2024):')
print(records[records['season'] == 2024].sort_values('win_rate', ascending=False).head(5))

Win rate sample (2024):
     season                   team  matches  wins  losses  win_rate
139    2024  Kolkata Knight Riders       14    11       3     0.786
143    2024       Rajasthan Royals       15     9       6     0.600
145    2024    Sunrisers Hyderabad       16     9       7     0.562
136    2024    Chennai Super Kings       14     7       7     0.500
137    2024         Delhi Capitals       14     7       7     0.500


## 6. Toss Decisions

For each team per season: how many times did they win the toss and choose to bat vs field? And what was the win rate in each scenario?

In [6]:
# Only rows where this team won the toss
toss = mat_df[mat_df['winner'].notna()][['season', 'toss_winner', 'toss_decision', 'team1', 'team2', 'winner']].copy()
toss = toss.rename(columns={'toss_winner': 'team'})

toss_stats = (
    toss.groupby(['season', 'team', 'toss_decision'])
    .size()
    .reset_index(name='toss_won_count')
)

# Pivot bat/field into columns
toss_wide = toss_stats.pivot_table(
    index=['season', 'team'],
    columns='toss_decision',
    values='toss_won_count',
    fill_value=0,
).reset_index()
toss_wide.columns.name = None

# Rename columns - handle case where bat or field column might not exist in older seasons
for col, new_col in [('bat', 'toss_chose_bat'), ('field', 'toss_chose_field')]:
    if col in toss_wide.columns:
        toss_wide = toss_wide.rename(columns={col: new_col})
    else:
        toss_wide[new_col] = 0

print('Toss decisions sample:')
print(toss_wide[toss_wide['season'] == 2024].head(5))

Toss decisions sample:
     season                   team  toss_chose_bat  toss_chose_field
136    2024    Chennai Super Kings           0.000             3.000
137    2024         Delhi Capitals           2.000             5.000
138    2024         Gujarat Titans           0.000             3.000
139    2024  Kolkata Knight Riders           1.000             2.000
140    2024   Lucknow Super Giants           4.000             5.000


## 7. Batting Depth

Lower-order contribution: average runs scored by batters at positions 7-9 per match. This tells us which teams have genuine batting depth vs teams that rely on their top 6.

In [7]:
# Filter to lower-order batters (positions 7-9), wides excluded
legal = del_df[del_df['is_wide'] == False]
lower_order = legal[legal['batting_position'] >= 7].copy()

depth = (
    lower_order.groupby(['season', 'match_id', 'batting_team'])['batter_runs']
    .sum()
    .reset_index(name='lower_order_runs')
    .groupby(['season', 'batting_team'])['lower_order_runs']
    .mean()
    .reset_index(name='depth_avg_runs')
    .rename(columns={'batting_team': 'team'})
)

print('Batting depth sample (2024):')
print(depth[depth['season'] == 2024].sort_values('depth_avg_runs', ascending=False).head(5))

Batting depth sample (2024):
     season                   team  depth_avg_runs
142    2024           Punjab Kings          37.750
145    2024    Sunrisers Hyderabad          37.091
139    2024  Kolkata Knight Riders          34.000
140    2024   Lucknow Super Giants          32.300
138    2024         Gujarat Titans          29.375


## 8. Merge and Save

In [8]:
team_stats = phase_wide
team_stats = team_stats.merge(records,    on=['season', 'team'], how='left')
team_stats = team_stats.merge(toss_wide,  on=['season', 'team'], how='left')
team_stats = team_stats.merge(depth,      on=['season', 'team'], how='left')

# Fill any missing toss/depth values with 0
for col in ['toss_chose_bat', 'toss_chose_field', 'depth_avg_runs']:
    if col in team_stats.columns:
        team_stats[col] = team_stats[col].fillna(0)

float_cols = [c for c in team_stats.columns if team_stats[c].dtype == float]
team_stats[float_cols] = team_stats[float_cols].round(4)

team_stats = team_stats.sort_values(['season', 'team']).reset_index(drop=True)

team_stats.to_csv('../data/processed/team_season_stats.csv', index=False)
print(f'Saved {len(team_stats):,} rows to data/processed/team_season_stats.csv')
print(f'Columns: {list(team_stats.columns)}')

Saved 166 rows to data/processed/team_season_stats.csv
Columns: ['season', 'team', 'death_avg_runs', 'mid_avg_runs', 'pp_avg_runs', 'matches', 'wins', 'losses', 'win_rate', 'toss_chose_bat', 'toss_chose_field', 'depth_avg_runs']


## Summary

**How the dashboard uses this file**:
- Phase scoring profile: filter by team + season range, compare team avg vs league avg
- Season win rate trend: filter by team, plot win_rate by season
- Toss preference: filter by team, plot toss_chose_bat vs toss_chose_field
- Batting depth: filter by team + season range, plot depth_avg_runs

**Saved output**: `data/processed/team_season_stats.csv`